In [ ]:
import os
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, Subset
import torch.nn as nn
import torch.optim as optim
import logging
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
import random

In [ ]:
# Configuração de Logging
logging.basicConfig(filename='process_log.log', level=logging.ERROR,
                    format='%(asctime)s %(levelname)s %(message)s')

class ECGDataset(Dataset):
    def __init__(self, base_dir):
        self.ecg_data = []
        self.labels = []
        self.prontuarios = []
        self.load_processed_beats(base_dir)
    
    def load_processed_beats(self, base_dir):
        for label in ['0.0', '3.0']:
            folder_path = os.path.join(base_dir, label)
            for prontuario_folder in os.listdir(folder_path):
                prontuario_path = os.path.join(folder_path, prontuario_folder)
                if os.path.isdir(prontuario_path):
                    for filename in os.listdir(prontuario_path):
                        if filename.endswith('.npy'):
                            file_path = os.path.join(prontuario_path, filename)
                            try:
                                data = np.load(file_path)
                                if data.size == 0:
                                    raise ValueError(f"Arquivo vazio: {file_path}")
                                self.ecg_data.append(data)
                                label_value = int(label.split('.')[0])
                                self.labels.append(0 if label_value == 0 else 1)
                                self.prontuarios.append(prontuario_folder)
                            except Exception as e:
                                logging.error(f"Erro ao carregar o arquivo {file_path}: {e}")
    
    def __len__(self):
        return len(self.ecg_data)
    
    def __getitem__(self, idx):
        signal = torch.tensor(self.ecg_data[idx], dtype=torch.float32).unsqueeze(0)  # Adicionar um canal de entrada
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        prontuario = self.prontuarios[idx]
        return signal, label, prontuario

In [9]:
# Definir diretório base
base_dir = "/home/leo-vitor/Documentos/UFC/biosinais/codigos/Dist_Cond_AtrioVent_Classification/Transformer-based_detection/processed_beats"

# Criar o dataset
dataset = ECGDataset(base_dir)

# Obter listas de índices para cada prontuário
prontuario_to_indices = {}
for idx, prontuario in enumerate(dataset.prontuarios):
    if prontuario not in prontuario_to_indices:
        prontuario_to_indices[prontuario] = []
    prontuario_to_indices[prontuario].append(idx)

# Dividir os prontuários em treino, validação e teste
train_prontuarios, temp_prontuarios = train_test_split(list(prontuario_to_indices.keys()), test_size=0.3, random_state=42)
val_prontuarios, test_prontuarios = train_test_split(temp_prontuarios, test_size=0.5, random_state=42)

# Converter listas de listas em listas simples de índices
train_indices = [idx for prontuario in train_prontuarios for idx in prontuario_to_indices[prontuario]]
val_indices = [idx for prontuario in val_prontuarios for idx in prontuario_to_indices[prontuario]]
test_indices = [idx for prontuario in test_prontuarios for idx in prontuario_to_indices[prontuario]]

# Criar subsets para cada conjunto
train_dataset = Subset(dataset, train_indices)
val_dataset = Subset(dataset, val_indices)
test_dataset = Subset(dataset, test_indices)

# Criar DataLoaders para treinamento, validação e teste
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [ ]:
# Definindo a arquitetura do modelo 1D-CNN
class CNN1D(nn.Module):
    def __init__(self):
        super(CNN1D, self).__init__()
        self.conv_block1 = self._conv_block(1, 128, kernel_size=10)
        self.conv_block2 = self._conv_block(128, 128, kernel_size=10)
        self.conv_block3 = self._conv_block(128, 128, kernel_size=10)
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(128 * 53, 512)
        self.dropout = nn.Dropout(0.2)
        self.fc2 = nn.Linear(512, 2)
    
    def _conv_block(self, in_channels, out_channels, kernel_size):
        return nn.Sequential(
            nn.Conv1d(in_channels, out_channels, kernel_size, padding=kernel_size // 2),
            nn.ReLU(),
            nn.Conv1d(out_channels, out_channels, kernel_size, padding=kernel_size // 2),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Dropout(0.2),
            nn.BatchNorm1d(out_channels)
        )
    
    def forward(self, x):
        x = self.conv_block1(x)
        x = self.conv_block2(x)
        x = self.conv_block3(x)
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x

# Treinando o modelo
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CNN1D().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-6)

num_epochs = 100

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for signals, labels, _ in train_loader:
        signals, labels = signals.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(signals)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for signals, labels, _ in val_loader:
            signals, labels = signals.to(device), labels.to(device)
            outputs = model(signals)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_loader)}, Val Loss: {val_loss/len(val_loader)}, Val Accuracy: {100 * correct / total}")

In [ ]:
# Avaliação no conjunto de teste
model.eval()
test_loss = 0.0
correct = 0
total = 0
with torch.no_grad():
    for signals, labels, _ in test_loader:
        signals, labels = signals.to(device), labels.to(device)
        signals = signals.unsqueeze(1)  # Adicionar um canal de entrada
        outputs = model(signals)
        loss = criterion(outputs, labels)
        test_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Test Loss: {test_loss/len(test_loader)}, Test Accuracy: {100 * correct / total}")